In [21]:
from pathlib import Path
import pandas as pd

# Define the path of your folder containing 60 zips
zip_folder = Path(r"C:\Users\ASUS\Desktop\tennis_data")

# Read proper table (MatchTimeInfo)
MatchTimeInfo_df = pd.read_parquet(zip_folder / "time_all.parquet")

# Make a copy and clean data
MatchTimeInfo_clean = MatchTimeInfo_df.copy()

# Drop period_4 and period_5 columns because they are totally empty
# Drop date column because it has brought duplicity
MatchTimeInfo_clean.drop(
    columns=["period_4", "period_5", "date"],
    inplace=True
)

# Drop duplicates because the files are snapshots
MatchTimeInfo_clean = (
    MatchTimeInfo_clean
    .drop_duplicates(subset="match_id", keep="last")
)

# Drop negative values (only 2 records are negative in period_2 column)
MatchTimeInfo_clean = MatchTimeInfo_clean[
    MatchTimeInfo_clean["period_2"] >= 0
].copy()

# Create total duration column
MatchTimeInfo_clean["match_duration"] = (
    MatchTimeInfo_clean["period_1"].fillna(0)
    + MatchTimeInfo_clean["period_2"].fillna(0)
    + MatchTimeInfo_clean["period_3"].fillna(0)
)

In [28]:
# Find the longest match with not deleting outliers
MatchTimeInfo_clean.sort_values("match_duration", ascending=False).head(1)

,match_id,period_1,period_2,period_3,current_period_start_timestamp,match_duration
7435,12063611,167352.0,169438.0,NaN,1.707823e+09,336790.0


In [24]:
# Delete outliers of periods
def remove_outliers(df, columns):
    df = df.copy()

    for col in columns:

        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        df = df[ df[col].isna() |
            ((df[col] >= lower) & (df[col] <= upper))
        ]

    return df

In [25]:
# 1st road: Clean table -> total duration -> drop out liers of periods
MatchTime_without_outliers1 = MatchTimeInfo_clean.copy()
MatchTime_without_outliers1 = remove_outliers(
    MatchTime_without_outliers1,
    ["period_1", "period_2", "period_3"]
)

MatchTime_without_outliers1.sort_values("match_duration", ascending=False).head(1)


,match_id,period_1,period_2,period_3,current_period_start_timestamp,match_duration
919,12024245,4564.0,3502.0,5138.0,1.706961e+09,13204.0


In [26]:
# 2nd road: Clean table -> total duration -> drop out liers of total duration
MatchTime_without_outliers2 = MatchTimeInfo_clean.copy()
MatchTime_without_outliers2 = remove_outliers(
    MatchTime_without_outliers2,
    ["match_duration"]
)

MatchTime_without_outliers2.sort_values("match_duration", ascending=False).head(1)

,match_id,period_1,period_2,period_3,current_period_start_timestamp,match_duration
30913,12181372,2517.0,5601.0,3758.0,1.711127e+09,11876.0


In [29]:
# 3rd road: Clean table -> total duration -> drop out liers of total duration and  periods
MatchTime_without_outliers3 = MatchTimeInfo_clean.copy()
MatchTime_without_outliers3 = remove_outliers(
    MatchTime_without_outliers3,
    ["period_1", "period_2", "period_3", "match_duration"]
)

MatchTime_without_outliers3.sort_values("match_duration", ascending=False).head(1)


,match_id,period_1,period_2,period_3,current_period_start_timestamp,match_duration
19832,12125055,3583.0,2738.0,4859.0,1.709739e+09,11180.0


In [ ]:
# Use a table to find players' name in a desired match
MatchEventInfo_df = pd.read_parquet(
    zip_folder / "event_all.parquet"
)
MatchEventInfo_df[MatchEventInfo_df["match_id"]==12024245]